# Clustering messages using BERTopic

## Installing dependencies

## Importing data and pre-processing

In [1]:

import pandas as pd
df_en = pd.read_excel("..\\..\\dataset\\original_datasets\\translated_messages.xlsx")
df_en.head(10)
df = df_en

In [ ]:
# number of unique patients
print("Number of unique patients: ", df_en["UserGUID"].nunique())

In [3]:
from bs4 import BeautifulSoup
def clean_html_with_soup(text):
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text()

df['Message'] = df['Message'].apply(lambda x: clean_html_with_soup(str(x)))

C:\Users\p70092940\AppData\Local\Temp\ipykernel_18988\2495750112.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, 'html.parser')


In [4]:
import re
set_anonymization = set()
pattern = r'\[\w+\-\d*]'  # Pattern to find anonymized words in the format [word]

def replace_numbered_patterns(text):
    # Match any pattern of the form [WORD-<number>] and replace with [WORD]
    return re.sub(r'\[(\w+)-\d+\]', r'[\1]', text)


# Apply the find_anonymized_words function to each message after cleaning HTML
df['Message'] = df['Message'].apply(lambda x: replace_numbered_patterns(str(x)))

df['Message']

0                                                  Message
1                                                      nan
2        Goedemiddag,  Dank voor uw aanmelding en het i...
3        Beste gebruikers van mijnIBDcoach,   Van 11 fe...
4        Beste gebruiker van mijnIBDcoach,   In verband...
                               ...                        
33234    A.U.B. NIET REAGEREN OP DIT BERICHT  Beste geb...
33235    A.U.B. NIET REAGEREN OP DIT BERICHT  Beste geb...
33236    AUB NIET REAGEREN OP DIT BERICHT  Beste gebrui...
33237    VRAGENLIJST FACTOREN UIT HET  DAGELIJKSE LEVEN...
33238    Dag,Uit onderzoek blijkt dat bepaalde factoren...
Name: Message, Length: 33239, dtype: object

In [9]:
# list of all all messages 

messages = df["Message"].tolist()
messages = messages[2:]

# check if html tags still exist
for message in messages:
    if '<br' in message:
        print(message)

In [9]:
# subset of messages from/to client, and with/without duplicates

messages_fromclient = df[df["richting (van of naar cliënt)"] == "From client"]["Message"].tolist() 	 
print('Number of messages from patients: ', len(messages_fromclient))
unique_messages_fromclient = list(set(messages_fromclient))
print('Number of unique messages from patients: ', len(unique_messages_fromclient))

messages_toclient = df[df["richting (van of naar cliënt)"] == "To client"]["Message"].tolist() 	 
print('Number of messages from coaches: ', len(messages_toclient))
unique_messages_toclient = list(set(messages_toclient))
print('Number of unique messages to patients: ', len(unique_messages_toclient))

Number of messages from patients:  13199
Number of unique messages from patients:  12631
Number of messages from coaches:  20038
Number of unique messages to patients:  11468


### Preprocessing the messages by removing stopwords

In [5]:
### Preprocess the messages by manually removing stopwords

with open('..\\..\\dataset\\stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

In [153]:
pd.set_option('display.max_colwidth', None)

## Embeddings

In [6]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
embeddings = embedding_model(messages_fromclient, truncation=True, padding=True)

In [17]:
import numpy as np
sentence_embeddings = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings])
np.save('..\\..\\saved_models\\MESSAGE_embeddings_bertje_cl.npy', sentence_embeddings)

In [7]:
#load embeddings
import numpy as np
embeddings = np.load('..\\..\\saved_models\\MESSAGE_embeddings_bertje_cl.npy')

## Running NLP Models

### Model 1. Multilingual Sentence Transformer

In [11]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42),
  hdbscan_model = HDBSCAN(min_cluster_size=70, min_samples = 70, cluster_selection_epsilon = 0.01, metric='euclidean', cluster_selection_method='eom', prediction_data=True),
  vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{2,}\b'),

  # Hyperparameters
  top_n_words=10,
  verbose=True,
  min_topic_size = 5,
  nr_topics=15
)

# Train model
topics, probs = topic_model.fit_transform(messages_fromclient, embeddings)

fig = topic_model.visualize_topics()
fig.show()

2024-10-08 09:26:22,241 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-10-08 09:26:43,528 - BERTopic - Dimensionality - Completed ✓
2024-10-08 09:26:43,528 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-10-08 09:26:44,261 - BERTopic - Cluster - Completed ✓
2024-10-08 09:26:44,261 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-10-08 09:26:44,530 - BERTopic - Representation - Completed ✓
2024-10-08 09:26:44,530 - BERTopic - Topic reduction - Reducing number of topics
2024-10-08 09:26:44,534 - BERTopic - Topic reduction - Reduced number of topics from 4 to 4
c:\Users\p70092940\OneDrive - Maastricht University\Desktop\Projects\MyIBDcoach-NLP\mijnidbcoachnlp\.venv\Lib\site-packages\umap\spectral.py:521: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(


TypeError: Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.